# 2 卷积和池化层

## 2.1 理论计算题

输入一张大小为 $3 \times 32 \times 32$（通道数 $\times$ 高 $\times$ 宽）的彩色图像。通过一个包含 16 个 $3 \times 5 \times 5$ 卷积核的卷积层，填充（Padding）为 2，步幅（Stride）为 2。

**1. 卷积层输出的特征图尺寸：**
计算公式为：
$$ H_{out} = \lfloor \frac{H_{in} - F_{H} + 2P}{S} \rfloor + 1 $$
$$ W_{out} = \lfloor \frac{W_{in} - F_{W} + 2P}{S} \rfloor + 1 $$
代入已知数值：
$$ H_{out} = \lfloor \frac{32 - 5 + 2 \times 2}{2} \rfloor + 1 = \lfloor \frac{31}{2} \rfloor + 1 = 15 + 1 = 16 $$
由于图像宽高对称，同理可得宽均为 16。
输出通道数等于卷积核的个数 16，因此**特征图的尺寸为 $16 \times 16 \times 16$**。

**2. 单个输出通道的一个像素值需要的点乘次数：**
由于不带偏置或无论偏置，一次卷积操作是对应一个输入“感受野”内的局部区域与某个特定的卷积核逐元素相乘。
该感受野包含的元素数为：$\text{通道数} \times F_H \times F_W = 3 \times 5 \times 5 = 75$。
因此，**需要对输入进行 75 次点乘（乘法）操作**。

## 2.2 编程题
不使用深度学习框架的底层 Pooling API，仅使用 Python 和 NumPy（或 PyTorch 基础张量操作），手动实现一个支持步幅（stride）和填充（padding）的二维最大池化（Max Pooling）前向传播函数。

In [7]:
import numpy as np

def max_pool2d(x, pool_size, stride=None, padding=0):
    """
    二维最大池化前向传播
    :param x: 输入张量，形状为 (C, H, W) 或 (N, C, H, W)
    :param pool_size: 池化窗口大小，整数或元组 (k_H, k_W)
    :param stride: 步幅，整数或元组 (s_H, s_W)，默认为 pool_size
    :param padding: 填充，整数或元组 (p_H, p_W)，默认为 0
    :return: 最大池化特征图
    """
    if isinstance(pool_size, int):
        pool_size = (pool_size, pool_size)
    if stride is None:
        stride = pool_size
    elif isinstance(stride, int):
        stride = (stride, stride)
    if isinstance(padding, int):
        padding = (padding, padding)
        
    is_3d = False
    if x.ndim == 3:
        is_3d = True
        x = x[np.newaxis, ...] # 转为 (N, C, H, W)
        
    N, C, H, W = x.shape
    k_H, k_W = pool_size
    s_H, s_W = stride
    p_H, p_W = padding
    
    # 对空间维度使用常数 -inf 进行填充，以保证最大池化不受 padding 为 0 的影响
    if p_H > 0 or p_W > 0:
        x_pad = np.pad(x, ((0, 0), (0, 0), (p_H, p_H), (p_W, p_W)), mode='constant', constant_values=-np.inf)
    else:
        x_pad = x
        
    # 计算输出特征图的尺寸
    H_out = int((H + 2 * p_H - k_H) / s_H) + 1
    W_out = int((W + 2 * p_W - k_W) / s_W) + 1
    
    # 预分配输出张量
    out = np.zeros((N, C, H_out, W_out))
    
    for i in range(H_out):
        for j in range(W_out):
            h_start = i * s_H
            h_end = h_start + k_H
            w_start = j * s_W
            w_end = w_start + k_W
            
            x_slice = x_pad[:, :, h_start:h_end, w_start:w_end]
            # 对感受野内的所有元素取全局最大值
            out[:, :, i, j] = np.max(x_slice, axis=(2, 3))
            
    if is_3d:
        return out[0]
    return out

# 测试代码
x = np.random.randn(3, 32, 32)
out = max_pool2d(x, pool_size=2, stride=2, padding=0)
print("Input shape:", x.shape)
print("Output shape:", out.shape)

Input shape: (3, 32, 32)
Output shape: (3, 16, 16)


# 3 LeNet, AlexNet, VGG 和 NiN

## 3.1 理论计算题

**1. 一个 $5 \times 5$ 卷积层（不带偏置）的参数量：**
由于输入和输出特征图通道数均为 $C$：
$$ \text{参数量} = C_{\text{in}} \times C_{\text{out}} \times F_H \times F_W = C \times C \times 5 \times 5 = 25C^2 $$

**2. 两个串联的 $3 \times 3$ 卷积层（不带偏置，通道数均为 $C$）的总参数量：**
由于是两层级联，每一层参数量为 $C \times C \times 3 \times 3 = 9C^2$：
$$ \text{总参数量} = 9C^2 + 9C^2 = 18C^2 $$
*(注：通过利用两个重叠的 3x3 卷积级联，可以在保持等效于 5x5 感受野的同时减少网络参数）*

## 3.2 编程题
使用 PyTorch 定义一个标准的 NiN 块 (NiN Block)。

In [8]:
import torch
import torch.nn as nn

def nin_block(in_channels, out_channels, kernel_size, stride, padding):
    """
    定义一个 NiN 块
    它由一个普通的卷积层以及两个随后的 1x1 卷积层级联组成。
    每层卷积后都需要紧跟一个 ReLU 激活层。
    """
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU()
    )

# 测试代码
blk = nin_block(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)
x = torch.randn(1, 3, 32, 32)
out = blk(x)
print("NiN Block output shape:", out.shape)

NiN Block output shape: torch.Size([1, 16, 32, 32])


# 4 Inception, 批量归一化和残差网络

## 4.1 理论计算题
小批量内的输出特征值为：$x_1=2, x_2=4, x_3=6, x_4=8$。  
已知缩放参数 $\gamma=2$，平移参数 $\beta=1$，常数 $\epsilon=0$。

1. 首先计算该小批量样本的均值 $\mu$：
$$ \mu = \frac{2 + 4 + 6 + 8}{4} = \frac{20}{4} = 5 $$

2. 计算该小批量样本的方差 $\sigma^2$：
$$ \sigma^2 = \frac{(2-5)^2 + (4-5)^2 + (6-5)^2 + (8-5)^2}{4} = \frac{9 + 1 + 1 + 9}{4} = 5 $$

3. 对数据进行标准化处理：
$$ \hat{x}_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}} = \frac{x_i - 5}{\sqrt{5}} $$

4. 经由 $\gamma$ 和 $\beta$ 的仿射变换得到最终输出：
$$ y_i = \gamma \hat{x}_i + \beta = 2 \times \frac{x_i - 5}{\sqrt{5}} + 1 $$

分别代入各项得到 $y_1, y_2, y_3, y_4$：
- $y_1 = 2 \times \frac{2-5}{\sqrt{5}} + 1 = 1 - \frac{6\sqrt{5}}{5} \approx -1.683$
- $y_2 = 2 \times \frac{4-5}{\sqrt{5}} + 1 = 1 - \frac{2\sqrt{5}}{5} \approx 0.106$
- $y_3 = 2 \times \frac{6-5}{\sqrt{5}} + 1 = 1 + \frac{2\sqrt{5}}{5} \approx 1.894$
- $y_4 = 2 \times \frac{8-5}{\sqrt{5}} + 1 = 1 + \frac{6\sqrt{5}}{5} \approx 3.683$

## 4.2 编程题
自定义一个残差块类 `Residual`。

In [9]:
import torch
import torch.nn as nn

class Residual(nn.Module):
    def __init__(self, input_channels, num_channels, use_1x1conv=False, strides=1):
        super(Residual, self).__init__()
        # 第一个 3x3 卷积层极其批归一化
        self.conv1 = nn.Conv2d(input_channels, num_channels, kernel_size=3, padding=1, stride=strides)
        self.bn1 = nn.BatchNorm2d(num_channels)
        
        # 第二个 3x3 卷积层极其批归一化
        self.conv2 = nn.Conv2d(num_channels, num_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(num_channels)
        
        # 可选的 1x1 卷积，用于改变输入通道数或改变维度形状，使能够按加法合并
        if use_1x1conv:
            self.conv3 = nn.Conv2d(input_channels, num_channels, kernel_size=1, stride=strides)
        else:
            self.conv3 = None

    def forward(self, X):
        Y = nn.functional.relu(self.bn1(self.conv1(X)))
        Y = self.bn2(self.conv2(Y))
        
        if self.conv3:
            X = self.conv3(X)
            
        # 残差连接按元素相加：f(x) + x
        Y += X
        return nn.functional.relu(Y)

# 测试代码
residual_blk = Residual(input_channels=3, num_channels=6, use_1x1conv=True, strides=2)
x = torch.randn(4, 3, 32, 32)
out = residual_blk(x)
print("Residual Block output shape:", out.shape)

Residual Block output shape: torch.Size([4, 6, 16, 16])


# 5 图像增广，微调和样式迁移

## 5.1 理论计算题

**1. 为什么对除了最终输出层之外的“底层特征提取层”设置较小的学习率或冻结，而对新初始化的“顶层输出层”设置较大的学习率？**
* **底层网络特征具有通用性：** 预训练模型在大型源数据集（如 ImageNet）上学习到的底层特征通常是边缘、颜色、纹理等通用的基本视觉特征。这些通用特征即使过渡到了不相关的图像分类任务也依然很好复用，因此不需要进行大幅度修改，甚至可以设置较小的学习率或者冻结以节省算力。
* **顶层网络负责特定任务：** 网络的顶层通常学习的是与特定任务紧密相关的语义或者抽象特征划分。针对新的目标数据集进行微调时，需要这输出层去拟合新特征分布，我们通过设置较大学习率使其快速收敛并适应目标任务。

**2. 如果目标数据集非常小且与源数据集非常相似，防止过拟合的微调策略？**
* 由于源数据与目标数据非常相似，预训练模型在这上面提取的特征可以直接平替。
* 由于目标数据集非常小，网络继续训练大量的参数极易发生过拟合现象。
* **微调策略：** 应当截断冻结前面的大量或者全部提取层参数，只重新替换/初始化并微调网络最后的几层（例如分类层或最后的全连接层）。并在输出层补充额外的防止过拟合手段（Dropout、权重衰减，或者对目标数据实施图像增广增加其丰富度等）。

## 5.2 编程题
创建一个组合图像增广管道（Pipeline）。

In [10]:
import torch
from torchvision import transforms

# 创建图像增广管道
pipeline = transforms.Compose([
    # 1. 随机裁剪面积比范围 (0.08, 1.0)，并且缩放到 224x224 像素
    transforms.RandomResizedCrop(size=(224, 224), scale=(0.08, 1.0)),
    
    # 2. 50% 的概率对图像进行水平翻转
    transforms.RandomHorizontalFlip(p=0.5),
    
    # 3. 随机改变亮度、对比度和饱和度，变化范围设为 0.5
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    
    # 4. 最终将图像转换为 PyTorch 张量
    transforms.ToTensor()
])

print("Pipeline Created:")
print(pipeline)

Pipeline Created:
Compose(
    RandomResizedCrop(size=(224, 224), scale=(0.08, 1.0), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=True)
    RandomHorizontalFlip(p=0.5)
    ColorJitter(brightness=(0.5, 1.5), contrast=(0.5, 1.5), saturation=(0.5, 1.5), hue=None)
    ToTensor()
)


# 6 目标检测，计算机视觉训练技巧

## 6.1 理论计算题

设两个真实和预测的框分别是：
- 真实框（GT）$A = [10, 10, 50, 50]$
- 预测框（Pred）$B = [30, 30, 70, 70]$

**1. 两者的相交区域 (Intersection)：**
- $x_{\text{left}} = \max(10, 30) = 30$
- $y_{\text{top}} = \max(10, 30) = 30$
- $x_{\text{right}} = \min(50, 70) = 50$
- $y_{\text{bottom}} = \min(50, 70) = 50$

相交区域的宽为 $\max(0, 50 - 30) = 20$，高为 $\max(0, 50 - 30) = 20$。
相交面积 $\text{Intersection} = 20 \times 20 = 400$。

**2. 两条框的并集面积 (Union)：**
- $\text{Area}(A) = (50-10) \times (50-10) = 40 \times 40 = 1600$
- $\text{Area}(B) = (70-30) \times (70-30) = 40 \times 40 = 1600$
总并集面积 $\text{Union} = \text{Area}(A) + \text{Area}(B) - \text{Intersection} = 1600 + 1600 - 400 = 2800$。

**3. 计算最终 IoU：**
$$ \text{IoU} = \frac{\text{Intersection}}{\text{Union}} = \frac{400}{2800} = \frac{1}{7} \approx 0.142857 $$
**IoU 准确值为 $\frac{1}{7}$。**

## 6.2 编程题
实现一个计算标签平滑后交叉熵损失的函数。

In [11]:
import torch
import torch.nn.functional as F

def label_smooth_cross_entropy(logits, targets, epsilon=0.1):
    """
    计算使用了标签平滑的交叉熵损失。
    :param logits: 模型的预测输出，形状为 (N, K)
    :param targets: 真实标签，从 0 到 K-1，形状为 (N)
    :param epsilon: 平滑因子
    """
    K = logits.size(-1)
    
    # 将类索引转换为 One-hot 编码
    one_hot = F.one_hot(targets, num_classes=K).float()
    
    # 严格按照作业图片中的公式：
    # 真实标签对应的目标概率从 1 变为 1 - \epsilon
    # 其余错误类别的概率从 0 变为 \epsilon / (K - 1)
    smoothed_labels = one_hot * (1.0 - epsilon) + (1.0 - one_hot) * epsilon / (K - 1)
    
    # 计算预测的 log 概率分布矩阵
    log_probs = F.log_softmax(logits, dim=-1)
    
    # 计算均值交叉熵损失：-(smoothed_labels * log_probs).sum(类别维度) 的样本均值
    loss = -(smoothed_labels * log_probs).sum(dim=-1).mean()
    
    return loss

# 测试代码
N, K = 4, 10
logits = torch.randn(N, K)
targets = torch.randint(0, K, (N,))

# 计算手动实现的标签平滑
loss_manual = label_smooth_cross_entropy(logits, targets, epsilon=0.1)

print(f"Manual Output: {loss_manual.item():.6f}")

Manual Output: 2.402973
